# RoadWatch pothole detector

Runs on Kaggle with **Accelerator: GPU T4 x2** and **Internet: On**.

Add these datasets as inputs before running:

- `juusos/rdd2022es` (RDD2022ES, dashcam-perspective road damage)
- `sabidrahman/pothole-cracks-and-openmanhole` (open manholes, so the model stops reporting storm drains)
- `roadwatch-negatives` (your dashcam frames the previous model false-fired on, with manhole covers labelled)

The first two are CC BY-NC-SA 4.0. Non-commercial, attribution required.

Three classes: `pothole`, `manhole` and `crack`. RDD2022ES ships every damage type in two
severity tiers, but the deep pothole tier holds only ~270 boxes dataset-wide, which is too
few to learn from or to measure. Tiers collapse, the three crack types share one class, and
severity is derived from box geometry in the app instead.

Use **Save Version -> Save & Run All** so this runs headless and you can close the tab.

In [ ]:
!pip install -q ultralytics

import os

# Ultralytics writes its settings and a wandb prompt somewhere on first import.
os.environ["YOLO_CONFIG_DIR"] = "/kaggle/working"
os.environ["WANDB_DISABLED"] = "true"

# Public repo, so no credentials needed. If you make it private, upload
# prep_dataset.py as a Kaggle dataset and point at that instead.
!rm -rf /kaggle/working/roadwatch
!git clone -q https://github.com/K-man1/roadwatch.git /kaggle/working/roadwatch

## Build the dataset

`/kaggle/input` is read-only and Ultralytics writes a `labels.cache` next to the labels
directory on the first epoch, so the dataset has to live somewhere writable. `/kaggle/temp`
is the right choice over `/kaggle/working`: both are writable, but working gets versioned
into the notebook output on every commit and you do not want to re-upload gigabytes of
images each run.

The prep script drops RDD2022ES's mirrored duplicate frames (Ultralytics already applies
`fliplr` during training, so keeping both halves doubles the epoch for nothing), folds
the severity tiers into `pothole` and `crack`, merges in damage-free manhole frames, and
adds the sorted harvest directories to train only.

`--background-frac 0.3` holds frames with no damage at all to 30% of the training images:
enough that plain road stays quiet, not so many that predicting nothing looks safe.

The harvest directories are looked up by name under `/kaggle/input`, so it does not matter
how Kaggle nests the uploaded archives.

Add `--countries United_States Czech` to test whether the Japanese and Indian road
surfaces help or hurt on New Jersey footage.

In [ ]:
!rm -rf /kaggle/temp/data
!python /kaggle/working/roadwatch/prep_dataset.py \
    --rdd /kaggle/input/rdd2022es/combined_annotatedv2 \
    --manhole /kaggle/input/pothole-cracks-and-openmanhole \
    --out /kaggle/temp/data \
    --background-frac 0.3 \
    --negatives /kaggle/input/roadwatch-negatives/negatives_3671a \
        /kaggle/input/roadwatch-negatives/negatives_3671b \
        /kaggle/input/roadwatch-negatives/negatives_3673

import yaml

DATA = "/kaggle/temp/data/data.yaml"
# Read the classes back rather than restating them, so this notebook cannot drift
# out of step with whatever the prep script actually wrote.
NAMES = list(yaml.safe_load(open(DATA))["names"].values())
print(open(DATA).read())

## Check the labels before you spend GPU hours on them

A silently wrong class remap looks exactly like a correct one until you plot it. Boxes
should sit on actual road damage, and manhole boxes on actual manhole covers.

In [ ]:
import random
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

COLORS = ["#ff6b35", "#2e86ab", "#8ac926"]

def label_for(image_path):
    return Path(str(image_path).replace("/images/", "/labels/")).with_suffix(".txt")

annotated = [p for p in sorted(Path("/kaggle/temp/data/images/train").iterdir())
             if label_for(p).read_text().strip()]
sample = random.Random(0).sample(annotated, 8)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, path in zip(axes.ravel(), sample):
    image = Image.open(path)
    width, height = image.size
    ax.imshow(image)
    ax.axis("off")
    for line in label_for(path).read_text().splitlines():
        cls, x, y, w, h = line.split()[:5]
        cls = int(cls)
        x, y, w, h = float(x) * width, float(y) * height, float(w) * width, float(h) * height
        ax.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h,
                                       fill=False, lw=2, edgecolor=COLORS[cls]))
        ax.text(x - w / 2, y - h / 2 - 4, NAMES[cls], color=COLORS[cls], fontsize=9)
plt.tight_layout()
plt.show()

## Train

`imgsz=960` rather than the usual 640 is the deliberate choice here. A pothole thirty
metres down the road is around 40px tall in a 1080p frame; at 640 that lands near YOLO's
stride-8 detection floor and the model simply cannot see it. More input resolution beats
more parameters when the objects are small and far away. It costs phone inference time,
so benchmark before committing.

Pick **T4 x2**, not P100. Kaggle's current container ships PyTorch 2.10 with CUDA 12.8,
which dropped support for the P100's sm_60 compute capability; training dies with
`no kernel image is available for execution on the device` the moment the model moves
onto the GPU. The T4 is sm_75 and works. `device=0` uses one of the two cards, so no
DDP is involved. The T4 also has tensor cores, which the P100 lacks, so `amp=True`
actually buys something here.

`batch=-1` is AutoBatch, which probes VRAM and targets about 60% use. If it OOMs mid-run,
pin it to 16.

With cracks labelled, the training set is several times the ~2.2k images of the previous
run, which took about 77 seconds per epoch. A full 150 epochs at that size would run past
Kaggle's 12-hour session limit, and the version would fail before the test and export
cells ran. `time=9` caps training at nine hours: Ultralytics measures epoch time and
recomputes the epoch count to fit, which replaces `epochs`, and `patience=30` still stops
early if it plateaus. The prep cell prints the real image counts, so check them against
this estimate.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
model.train(
    data=DATA,
    epochs=150,
    imgsz=960,
    batch=-1,
    device=0,
    workers=4,
    patience=30,
    time=9,
    seed=0,
    project="/kaggle/working/runs",
    name="roadwatch",
)

## Per-class metrics on the held-out test split

Read these separately rather than trusting the overall mAP. Watch `manhole` in
particular: those images come from an augmented, non-dashcam source, so a strong manhole
score may reflect the model keying on image style rather than on what a manhole looks
like through a windshield.

In [ ]:
metrics = model.val(data=DATA, split="test")

print(f"{'class':10s} {'P':>7s} {'R':>7s} {'mAP50':>7s} {'mAP50-95':>9s}")
for i, name in enumerate(NAMES):
    precision, recall, ap50, ap = metrics.box.class_result(i)
    print(f"{name:10s} {precision:7.3f} {recall:7.3f} {ap50:7.3f} {ap:9.3f}")

## Pick the auto-report confidence threshold

This is a product decision, not a metric. Filing a false report with the town is far more
costly than missing one pothole, so the threshold that maximises F1 is probably not the
one you want. Read the precision column and pick the lowest confidence that still keeps
precision high enough that you would defend the report.

Each row is a full validation pass, so this cell takes a few minutes.

In [ ]:
import pandas as pd

rows = []
for conf in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7]:
    swept = model.val(data=DATA, split="val", conf=conf, plots=False, verbose=False)
    precision, recall, _, _ = swept.box.class_result(NAMES.index("pothole"))
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    rows.append({"conf": conf, "precision": precision, "recall": recall, "f1": f1})

print(pd.DataFrame(rows).to_string(index=False))

## Export for the phone

ncnn is the runtime you were already targeting. Export at the resolution you intend to
run at: exporting at 960 and then feeding it 640 frames throws away the reason you
trained at 960 in the first place, but 960 on a phone may not hold 30fps. Benchmark both.

`best.pt` is copied to the notebook output root so you can download it without digging
through the runs directory.

In [ ]:
import shutil

best = "/kaggle/working/runs/roadwatch/weights/best.pt"
YOLO(best).export(format="ncnn", imgsz=960)
shutil.copy(best, "/kaggle/working/roadwatch_best.pt")
print("done")